In [5]:
!pip install -q ultralytics pandas pillow tqdm ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.1 MB/s eta 0:00:00


In [1]:
import os
import json
import zipfile
import warnings
import pandas as pd
import torch

from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFile
from ultralytics import YOLO
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

In [2]:
from google.colab import files
from pathlib import Path
import zipfile
import os

print("Upload these two files:")
print("1. ground_truth.csv")
print("2. images.zip")

uploaded = files.upload()

uploaded_files = list(uploaded.keys())
print("Uploaded:", uploaded_files)

csv_candidates = [f for f in uploaded_files if f.lower().endswith(".csv")]
zip_candidates = [f for f in uploaded_files if f.lower().endswith(".zip")]

if len(csv_candidates) == 0:
    raise ValueError("No CSV uploaded. Please upload ground_truth.csv")

if len(zip_candidates) == 0:
    raise ValueError("No ZIP uploaded. Please upload images.zip")

CSV_FILE = csv_candidates[0]
ZIP_FILE = zip_candidates[0]

CSV_PATH = Path(CSV_FILE)
EXTRACT_DIR = Path("uploaded_images")
EXTRACT_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("CSV_PATH:", CSV_PATH)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("Extracted items:", os.listdir(EXTRACT_DIR)[:10])

Upload these two files:
1. ground_truth.csv
2. images.zip


Saving ground_truth.csv to ground_truth.csv
Saving proct.zip to proct.zip
Uploaded: ['ground_truth.csv', 'proct.zip']
CSV_PATH: ground_truth.csv
EXTRACT_DIR: uploaded_images
Extracted items: ['downloaded_images']


In [3]:
from pathlib import Path
import os

# Try to find the real image folder automatically
subdirs = [p for p in EXTRACT_DIR.rglob("*") if p.is_dir()]
candidate_dirs = [EXTRACT_DIR] + subdirs

IMAGE_EXTS = [".webp", ".jpg", ".jpeg", ".png"]

best_dir = None
best_count = -1

for d in candidate_dirs:
    count = sum(1 for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    if count > best_count:
        best_count = count
        best_dir = d

IMAGE_DIR = best_dir

print("Selected IMAGE_DIR:", IMAGE_DIR)
print("Image count found:", best_count)

sample_files = [p.name for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS][:10]
print("Sample image files:", sample_files)

Selected IMAGE_DIR: uploaded_images/downloaded_images
Image count found: 8926
Sample image files: ['7300.webp', '1209.webp', '375.webp', '1074.webp', '2339.webp', '5091.webp', '7311.webp', '3887.webp', '4399.webp', '4328.webp']


In [4]:
# =========================
# CONFIGURATION
# =========================

OUTPUT_DIR = Path("curated_dataset")
VISUALIZATION_DIR = Path("visualized_predictions")
LOG_JSON_PATH = Path("output.json")
LOG_CSV_PATH = Path("output_log.csv")

INTERNAL_CLASSES = {"person": 0, "laptop": 1, "cellphone": 2}
CLASS_NAMES = list(INTERNAL_CLASSES.keys())

# class-specific thresholds
CLASS_THRESHOLDS = {
    "person": 0.25,
    "laptop": 0.25,
    "cellphone": 0.15,
}

# phone sanity checks
PHONE_MIN_ABS_W = 8
PHONE_MIN_ABS_H = 8
PHONE_MIN_AREA_RATIO = 0.00003
PHONE_MAX_AREA_RATIO = 0.20
PHONE_MAX_ASPECT_RATIO = 4.0

# duplicate suppression
DUP_IOU_THRESHOLD = 0.70

# quick test option
BENCHMARK_ROWS = None  # set to 100 for faster testing

print("CSV exists:", CSV_PATH.exists())
print("IMAGE_DIR exists:", IMAGE_DIR.exists())

CSV exists: True
IMAGE_DIR exists: True


In [5]:
# =========================
# LOAD INPUT DATA
# =========================

df_input = pd.read_csv(CSV_PATH)
df_input.columns = df_input.columns.str.strip()

df_input = df_input.rename(columns={
    "Person": "person",
    "Laptop": "laptop",
    "cell_phone": "cell_phone"
})

required_cols = ["id", "person", "laptop", "cell_phone"]
missing = [c for c in required_cols if c not in df_input.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}\nFound: {df_input.columns.tolist()}")

# Fix 1.0 -> 1
df_input["id"] = pd.to_numeric(df_input["id"], errors="coerce")
df_input = df_input.dropna(subset=["id"]).copy()
df_input["id"] = df_input["id"].astype(int).astype(str)

df_input["person_gt"] = pd.to_numeric(df_input["person"], errors="coerce").fillna(0).astype(int)
df_input["laptop_gt"] = pd.to_numeric(df_input["laptop"], errors="coerce").fillna(0).astype(int)
df_input["cellphone_gt"] = pd.to_numeric(df_input["cell_phone"], errors="coerce").fillna(0).astype(int)

if BENCHMARK_ROWS is not None:
    df_eval = df_input.head(BENCHMARK_ROWS).copy()
else:
    df_eval = df_input.copy()

print("Rows loaded:", len(df_input))
print("Sample ids:", df_input["id"].head(10).tolist())
df_input.head()

Rows loaded: 8926
Sample ids: ['1', '3', '4', '5', '8', '9', '10', '11', '12', '13']


,id,person,laptop,cell_phone,Unnamed: 4,person_gt,laptop_gt,cellphone_gt
0,1,1,0,0,NaN,1,0,0
1,3,1,0,0,NaN,1,0,0
2,4,1,0,0,NaN,1,0,0
3,5,1,0,0,NaN,1,0,0
4,8,1,0,0,NaN,1,0,0


In [6]:
# =========================
# IMAGE LOADER
# =========================

def open_local_image(image_id, image_dir=IMAGE_DIR, exts=IMAGE_EXTS):
    image_id = str(image_id).strip()

    # exact match
    for ext in exts:
        path = image_dir / f"{image_id}{ext}"
        if path.exists():
            return Image.open(path).convert("RGB"), str(path)

    # fallback if something still comes as float-like
    try:
        clean_id = str(int(float(image_id)))
        for ext in exts:
            path = image_dir / f"{clean_id}{ext}"
            if path.exists():
                return Image.open(path).convert("RGB"), str(path)
    except Exception:
        pass

    return None, None

In [7]:
# =========================
# MODELS
# =========================

models = {
    "yolov8n": YOLO("yolov8n.pt"),
    "yolo26": YOLO("yolo26n.pt"),
    "yolo_world": YOLO("yolov8x-worldv2.pt")
}

models["yolo_world"].set_classes(["person", "laptop", "cell phone"])

print("Loaded models:", list(models.keys()))

requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 36 packages in 771ms
Prepared 2 packages in 2.43s
Installed 2 packages in 2ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@b0c7af36eb99a5e103713e1792fc642f78059c39)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 3.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 179MiB/s]


Loaded models: ['yolov8n', 'yolo26', 'yolo_world']


In [8]:
# =========================
# LABEL + BOX HELPERS
# =========================

LABEL_MAP = {
    "person": "person",
    "laptop": "laptop",
    "cell phone": "cellphone",
    "mobile phone": "cellphone",
    "cellphone": "cellphone",
    "phone": "cellphone",
}

COLOR_MAP = {
    "person": "red",
    "laptop": "blue",
    "cellphone": "lime",
}

def normalize_label(name):
    if name is None:
        return None
    return LABEL_MAP.get(str(name).strip().lower())

def box_area(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def box_wh(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1), max(0.0, y2 - y1)

def compute_iou(box1, box2):
    x11, y11, x12, y12 = box1
    x21, y21, x22, y22 = box2

    xi1, yi1 = max(x11, x21), max(y11, y21)
    xi2, yi2 = min(x12, x22), min(y12, y22)

    inter_w = max(0.0, xi2 - xi1)
    inter_h = max(0.0, yi2 - yi1)
    inter = inter_w * inter_h

    area1 = box_area(box1)
    area2 = box_area(box2)
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0.0

def is_valid_phone_box(box, image_size):
    w, h = box_wh(box)
    img_w, img_h = image_size
    area_ratio = box_area(box) / max(1.0, img_w * img_h)

    if w < PHONE_MIN_ABS_W or h < PHONE_MIN_ABS_H:
        return False

    if area_ratio < PHONE_MIN_AREA_RATIO or area_ratio > PHONE_MAX_AREA_RATIO:
        return False

    aspect_ratio = max(w / max(h, 1e-6), h / max(w, 1e-6))
    if aspect_ratio > PHONE_MAX_ASPECT_RATIO:
        return False

    return True

In [9]:
# =========================
# FILTERING
# =========================

def deduplicate_boxes(detections, iou_threshold=DUP_IOU_THRESHOLD):
    detections = sorted(detections, key=lambda d: d["confidence"], reverse=True)
    kept = []

    for det in detections:
        keep = True
        for prev in kept:
            if det["class_name"] == prev["class_name"]:
                if compute_iou(det["box"], prev["box"]) >= iou_threshold:
                    keep = False
                    break
        if keep:
            kept.append(det)

    return kept

def apply_class_filters(detections, image_size):
    filtered = []

    for det in detections:
        cls = det["class_name"]
        conf = det["confidence"]
        box = det["box"]

        min_conf = CLASS_THRESHOLDS.get(cls, 0.25)
        if conf < min_conf:
            continue

        if cls == "cellphone" and not is_valid_phone_box(box, image_size):
            continue

        filtered.append(det)

    return deduplicate_boxes(filtered)

def detections_to_counts(detections):
    counts = {name: 0 for name in CLASS_NAMES}
    for det in detections:
        cls = det["class_name"]
        if cls in counts:
            counts[cls] += 1
    return counts

In [10]:
# =========================
# PREDICTION
# =========================

def predict_model(image, model_instance):
    raw_detections = []

    results = model_instance.predict(source=image, save=False, verbose=False)

    for res in results:
        names = res.names
        for cls_id, conf, bbox in zip(res.boxes.cls, res.boxes.conf, res.boxes.xyxy):
            label = names[int(cls_id)]
            class_name = normalize_label(label)

            if class_name is None:
                continue

            raw_detections.append({
                "class_name": class_name,
                "confidence": float(conf.item()),
                "box": [float(x) for x in bbox.tolist()],
            })

    filtered = apply_class_filters(raw_detections, image.size)
    counts = detections_to_counts(filtered)

    return {"counts": counts, "detections": filtered}

In [11]:
# =========================
# BENCHMARK
# =========================

benchmark_rows = []

for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Benchmarking"):
    image_id = row["id"]
    image, image_path = open_local_image(image_id)

    if image is None:
        benchmark_rows.append({
            "image_id": image_id,
            "model_name": "__image_not_found__",
            "image_error": f"No file found for {image_id} in {IMAGE_DIR}"
        })
        continue

    gt = {
        "person": int(row["person_gt"]),
        "laptop": int(row["laptop_gt"]),
        "cellphone": int(row["cellphone_gt"]),
    }

    for model_name, model in models.items():
        result = predict_model(image, model)
        pred = result["counts"]

        person_correct = int((pred["person"] > 0) == (gt["person"] > 0))
        laptop_correct = int((pred["laptop"] > 0) == (gt["laptop"] > 0))
        phone_correct = int((pred["cellphone"] > 0) == (gt["cellphone"] > 0))

        benchmark_rows.append({
            "image_id": image_id,
            "image_path": image_path,
            "model_name": model_name,
            "person_gt": gt["person"],
            "laptop_gt": gt["laptop"],
            "cellphone_gt": gt["cellphone"],
            "person_pred": pred["person"],
            "laptop_pred": pred["laptop"],
            "cellphone_pred": pred["cellphone"],
            "person_correct": person_correct,
            "laptop_correct": laptop_correct,
            "phone_correct": phone_correct,
            "overall_correct": int(person_correct and laptop_correct and phone_correct),
        })

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.head()

Benchmarking: 100%|██████████| 8926/8926 [11:31<00:00, 12.91it/s]


,image_id,image_path,model_name,person_gt,laptop_gt,cellphone_gt,person_pred,laptop_pred,cellphone_pred,person_correct,laptop_correct,phone_correct,overall_correct
0,1,uploaded_images/downloaded_images/1.webp,yolov8n,1,0,0,1,0,1,1,1,0,0
1,1,uploaded_images/downloaded_images/1.webp,yolo26,1,0,0,1,0,0,1,1,1,1
2,1,uploaded_images/downloaded_images/1.webp,yolo_world,1,0,0,1,0,0,1,1,1,1
3,3,uploaded_images/downloaded_images/3.webp,yolov8n,1,0,0,1,0,1,1,1,0,0
4,3,uploaded_images/downloaded_images/3.webp,yolo26,1,0,0,1,0,0,1,1,1,1


In [12]:
valid_benchmark_df = benchmark_df[benchmark_df["model_name"] != "__image_not_found__"].copy()

summary = (
    valid_benchmark_df
    .groupby("model_name")
    .agg(
        images=("image_id", "count"),
        person_accuracy=("person_correct", "mean"),
        laptop_accuracy=("laptop_correct", "mean"),
        phone_accuracy=("phone_correct", "mean"),
        overall_accuracy=("overall_correct", "mean"),
    )
    .reset_index()
)

for col in ["person_accuracy", "laptop_accuracy", "phone_accuracy", "overall_accuracy"]:
    summary[col] = (summary[col] * 100).round(2)

summary["score"] = (
    summary["phone_accuracy"] * 2 +
    summary["person_accuracy"] +
    summary["laptop_accuracy"]
)

summary = summary.sort_values(by="score", ascending=False).reset_index(drop=True)

display(summary)

BEST_MODEL_NAME = summary.iloc[0]["model_name"]
BEST_MODEL = models[BEST_MODEL_NAME]

print("Selected best global model:", BEST_MODEL_NAME)

,model_name,images,person_accuracy,laptop_accuracy,phone_accuracy,overall_accuracy,score
0,yolo_world,8926,96.98,92.95,93.82,85.10,377.57
1,yolo26,8926,91.33,97.93,93.52,83.46,376.30
2,yolov8n,8926,90.72,97.14,92.16,80.65,372.18


Selected best global model: yolo_world


In [13]:
# =========================
# TRUST LOGIC
# =========================

def evaluate_prediction_trust(gt, pred):
    person_correct = int((pred["person"] > 0) == (gt["person"] > 0))
    laptop_correct = int((pred["laptop"] > 0) == (gt["laptop"] > 0))
    phone_correct = int((pred["cellphone"] > 0) == (gt["cellphone"] > 0))

    overall_correct = int(person_correct and laptop_correct and phone_correct)

    if overall_correct == 1:
        return {
            "status": "perfect_match",
            "person_correct": person_correct,
            "laptop_correct": laptop_correct,
            "phone_correct": phone_correct,
            "overall_correct": overall_correct,
        }

    if gt["cellphone"] == 0 and pred["cellphone"] == 0 and person_correct == 1 and laptop_correct == 1:
        return {
            "status": "trusted_no_phone",
            "person_correct": person_correct,
            "laptop_correct": laptop_correct,
            "phone_correct": phone_correct,
            "overall_correct": overall_correct,
        }

    return {
        "status": "review_needed",
        "person_correct": person_correct,
        "laptop_correct": laptop_correct,
        "phone_correct": phone_correct,
        "overall_correct": overall_correct,
    }

In [14]:
# =========================
# SAVE HELPERS
# =========================

def convert_to_yolo_format(box, img_size):
    img_w, img_h = img_size
    x1, y1, x2, y2 = box

    x_c = (x1 + x2) / 2.0
    y_c = (y1 + y2) / 2.0
    w = x2 - x1
    h = y2 - y1

    return f"{x_c/img_w:.6f} {y_c/img_h:.6f} {w/img_w:.6f} {h/img_h:.6f}"

def create_labeled_image(image, detections):
    labeled_image = image.copy().convert("RGB")
    draw = ImageDraw.Draw(labeled_image)

    try:
        font = ImageFont.truetype("arial.ttf", size=20)
    except IOError:
        font = ImageFont.load_default()

    for det in detections:
        class_name = det["class_name"]
        box_coords = det["box"]
        conf = det["confidence"]
        color = COLOR_MAP.get(class_name, "white")

        draw.rectangle(box_coords, outline=color, width=3)
        label = f"{class_name} {conf:.2f}"

        text_pos = (box_coords[0] + 2, max(0, box_coords[1] - 20))
        text_bbox = draw.textbbox(text_pos, label, font=font)
        draw.rectangle(text_bbox, fill=color)
        draw.text(text_pos, label, fill="black", font=font)

    return labeled_image

def save_data(status, image_id, image, detections, model_name):
    base_dir = TRUSTED_DIR if status in {"perfect_match", "trusted_no_phone"} else REVIEW_DIR
    model_dir = base_dir / model_name

    images_dir = model_dir / "images"
    labels_dir = model_dir / "labels"
    viz_dir = VISUALIZATION_DIR / status / model_name

    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    viz_dir.mkdir(parents=True, exist_ok=True)

    filename_base = f"{image_id}_{model_name}"

    image_path = images_dir / f"{filename_base}.jpg"
    label_path = labels_dir / f"{filename_base}.txt"
    viz_path = viz_dir / f"{filename_base}.png"

    image.convert("RGB").save(image_path, "JPEG")

    with open(label_path, "w") as f:
        for det in detections:
            class_id = INTERNAL_CLASSES[det["class_name"]]
            yolo_box = convert_to_yolo_format(det["box"], image.size)
            f.write(f"{class_id} {yolo_box}\n")

    labeled_image = create_labeled_image(image, detections)
    labeled_image.save(viz_path)

    return {
        "image_path_saved": str(image_path),
        "yolo_annotation_path": str(label_path),
        "visualization_path": str(viz_path),
    }

In [15]:
# =========================
# FINAL ANNOTATION PASS
# =========================

TRUSTED_DIR = OUTPUT_DIR / "trusted"
REVIEW_DIR = OUTPUT_DIR / "review"

processing_log = []

for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=f"Annotating with {BEST_MODEL_NAME}"):
    image_id = row["id"]
    image, image_path = open_local_image(image_id)

    log_entry = {
        "id": image_id,
        "best_model": BEST_MODEL_NAME,
    }

    if image is None:
        log_entry["status"] = "image_not_found"
        processing_log.append(log_entry)
        continue

    gt = {
        "person": int(row["person_gt"]),
        "laptop": int(row["laptop_gt"]),
        "cellphone": int(row["cellphone_gt"]),
    }

    result = predict_model(image, BEST_MODEL)
    pred = result["counts"]
    detections = result["detections"]

    trust = evaluate_prediction_trust(gt, pred)

    log_entry.update({
        "source_image_path": image_path,
        "ground_truth": gt,
        "predicted_counts": pred,
        "person_correct": trust["person_correct"],
        "laptop_correct": trust["laptop_correct"],
        "phone_correct": trust["phone_correct"],
        "overall_correct": trust["overall_correct"],
        "status": trust["status"],
        "num_boxes_saved": len(detections),
    })

    saved_paths = save_data(trust["status"], image_id, image, detections, BEST_MODEL_NAME)
    log_entry.update(saved_paths)

    processing_log.append(log_entry)

log_df = pd.DataFrame(processing_log)
log_df.to_csv(LOG_CSV_PATH, index=False)

with open(LOG_JSON_PATH, "w") as f:
    json.dump(processing_log, f, indent=4)

print("Saved CSV log:", LOG_CSV_PATH)
print("Saved JSON log:", LOG_JSON_PATH)
log_df.head()

Annotating with yolo_world: 100%|██████████| 8926/8926 [13:31<00:00, 11.00it/s]


Saved CSV log: output_log.csv
Saved JSON log: output.json


,id,best_model,source_image_path,ground_truth,predicted_counts,person_correct,laptop_correct,phone_correct,overall_correct,status,num_boxes_saved,image_path_saved,yolo_annotation_path,visualization_path
0,1,yolo_world,uploaded_images/downloaded_images/1.webp,"{'person': 1, 'laptop': 0, 'cellphone': 0}","{'person': 1, 'laptop': 0, 'cellphone': 0}",1,1,1,1,perfect_match,1,curated_dataset/trusted/yolo_world/images/1_yo...,curated_dataset/trusted/yolo_world/labels/1_yo...,visualized_predictions/perfect_match/yolo_worl...
1,3,yolo_world,uploaded_images/downloaded_images/3.webp,"{'person': 1, 'laptop': 0, 'cellphone': 0}","{'person': 1, 'laptop': 0, 'cellphone': 0}",1,1,1,1,perfect_match,1,curated_dataset/trusted/yolo_world/images/3_yo...,curated_dataset/trusted/yolo_world/labels/3_yo...,visualized_predictions/perfect_match/yolo_worl...
2,4,yolo_world,uploaded_images/downloaded_images/4.webp,"{'person': 1, 'laptop': 0, 'cellphone': 0}","{'person': 1, 'laptop': 0, 'cellphone': 0}",1,1,1,1,perfect_match,1,curated_dataset/trusted/yolo_world/images/4_yo...,curated_dataset/trusted/yolo_world/labels/4_yo...,visualized_predictions/perfect_match/yolo_worl...
3,5,yolo_world,uploaded_images/downloaded_images/5.webp,"{'person': 1, 'laptop': 0, 'cellphone': 0}","{'person': 2, 'laptop': 0, 'cellphone': 0}",1,1,1,1,perfect_match,2,curated_dataset/trusted/yolo_world/images/5_yo...,curated_dataset/trusted/yolo_world/labels/5_yo...,visualized_predictions/perfect_match/yolo_worl...
4,8,yolo_world,uploaded_images/downloaded_images/8.webp,"{'person': 1, 'laptop': 0, 'cellphone': 0}","{'person': 3, 'laptop': 0, 'cellphone': 0}",1,1,1,1,perfect_match,3,curated_dataset/trusted/yolo_world/images/8_yo...,curated_dataset/trusted/yolo_world/labels/8_yo...,visualized_predictions/perfect_match/yolo_worl...


In [16]:
status_summary = log_df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count")
display(status_summary)

class_accuracy_summary = pd.DataFrame([{
    "person_accuracy_percent": round(log_df["person_correct"].dropna().mean() * 100, 2),
    "laptop_accuracy_percent": round(log_df["laptop_correct"].dropna().mean() * 100, 2),
    "phone_accuracy_percent": round(log_df["phone_correct"].dropna().mean() * 100, 2),
    "overall_accuracy_percent": round(log_df["overall_correct"].dropna().mean() * 100, 2),
}])
display(class_accuracy_summary)

review_df = log_df[log_df["status"] == "review_needed"].copy()
print("Review queue size:", len(review_df))
review_df.head()

,status,count
0,perfect_match,7596
1,review_needed,1330


,person_accuracy_percent,laptop_accuracy_percent,phone_accuracy_percent,overall_accuracy_percent
0,96.98,92.95,93.82,85.1


Review queue size: 1330


,id,best_model,source_image_path,ground_truth,predicted_counts,person_correct,laptop_correct,phone_correct,overall_correct,status,num_boxes_saved,image_path_saved,yolo_annotation_path,visualization_path
22,26,yolo_world,uploaded_images/downloaded_images/26.webp,"{'person': 1, 'laptop': 0, 'cellphone': 0}","{'person': 1, 'laptop': 0, 'cellphone': 1}",1,1,0,0,review_needed,2,curated_dataset/review/yolo_world/images/26_yo...,curated_dataset/review/yolo_world/labels/26_yo...,visualized_predictions/review_needed/yolo_worl...
27,31,yolo_world,uploaded_images/downloaded_images/31.webp,"{'person': 4, 'laptop': 0, 'cellphone': 0}","{'person': 5, 'laptop': 4, 'cellphone': 0}",1,0,1,0,review_needed,9,curated_dataset/review/yolo_world/images/31_yo...,curated_dataset/review/yolo_world/labels/31_yo...,visualized_predictions/review_needed/yolo_worl...
56,60,yolo_world,uploaded_images/downloaded_images/60.webp,"{'person': 1, 'laptop': 0, 'cellphone': 0}","{'person': 3, 'laptop': 2, 'cellphone': 0}",1,0,1,0,review_needed,5,curated_dataset/review/yolo_world/images/60_yo...,curated_dataset/review/yolo_world/labels/60_yo...,visualized_predictions/review_needed/yolo_worl...
57,61,yolo_world,uploaded_images/downloaded_images/61.webp,"{'person': 1, 'laptop': 0, 'cellphone': 0}","{'person': 3, 'laptop': 1, 'cellphone': 0}",1,0,1,0,review_needed,4,curated_dataset/review/yolo_world/images/61_yo...,curated_dataset/review/yolo_world/labels/61_yo...,visualized_predictions/review_needed/yolo_worl...
58,63,yolo_world,uploaded_images/downloaded_images/63.webp,"{'person': 2, 'laptop': 0, 'cellphone': 0}","{'person': 2, 'laptop': 1, 'cellphone': 0}",1,0,1,0,review_needed,3,curated_dataset/review/yolo_world/images/63_yo...,curated_dataset/review/yolo_world/labels/63_yo...,visualized_predictions/review_needed/yolo_worl...
